<a href="https://colab.research.google.com/github/abdulsamadkhan/AlignmentTuning/blob/main/DPO_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Direct Preference Optimization (DPO) Using Quantization



The goal of this tutorial is to develop a practical understanding of the Direct Preference Optimization (DPO) method and how to implement it using the `trl` library.

Throughout the tutorial, you'll gain hands-on experience in preparing a dataset formatted for DPO, applying the optimization process, and evaluating improvements in the performance of large language models (LLMs).

For this lab, we will specifically be using **GPT-2** as the base model for training with DPO.


## __Table of Contents__

<ol>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Importing-required-libraries">Importing required libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#Create-and-configure-the-model-and-tokenizer">Create and configure the model and tokenizer</a>
    </li>
    <li>
        <a href="#Quantized-model-configuration">Quantized model configuration</a>
    </li>
    <li>
        <a href="#Preprocess-dataset">Preprocess dataset</a>
    </li>
    <li>
        <a href="#DPO-configuration">DPO configuration</a>
    </li>
    <li>
        <a href="#DPO-training">DPO training</a>
    </li>
    <li>
        <a href="#Training">Training</a>
    </li>
    <li>
        <a href="#Generation">Generation</a>
    </li>
</ol>


----


#1. Setup


##1.1 Installing required libraries


In [ ]:
!pip install torch==2.3.1
!pip install --user trl==0.11.4 # for optimization training
!pip install peft==0.14.0 # for creating LoRA architecture
!pip install matplotlib==3.9.0
!pip install pandas
!pip install numpy==1.26.0
!pip install --user datasets==3.2.0

##1.2 Importing required libraries



In [ ]:
##imports
import multiprocessing
import os
import requests
import tarfile
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import load_dataset

from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer,TrainingArguments, GPT2Tokenizer, set_seed, GenerationConfig
from trl import DPOConfig, DPOTrainer


#2. Create and configure the model and tokenizer


In [ ]:
# Initialize the GPT-2 language model
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Initialize a separate reference model for comparison
model_ref = AutoModelForCausalLM.from_pretrained("gpt2")

# Initialize the tokenizer associated with GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Assign the end-of-sequence token as the padding token
tokenizer.pad_token = tokenizer.eos_token

# Align padding to the right side to prevent FP16 training overflow issues
tokenizer.padding_side = "right"

# Turn off caching in the model's forward method to ensure compatibility with some training setups
model.config.use_cache = False


In [ ]:
#Here, you can check the model architecture.
model

#3. Quantized model configuration


In [ ]:
# ⚙️ SETUP: Install compatible packages
!pip uninstall -y bitsandbytes
!pip install bitsandbytes --no-cache-dir
!pip install "transformers==4.45.2"
!pip install "accelerate>=0.27.2"

# 🧠 VERIFY GPU IS ENABLED
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)



This code sets up a 4-bit quantized version of GPT-2 using the BitsAndBytesConfig, allowing for efficient fine-tuning on GPU. It loads both the model and a reference model in quantized format and prepares the tokenizer with appropriate padding. Finally, it disables caching to ensure compatibility with training workflows like DPO.


In [ ]:
## Quantized model – requires GPU support

from transformers import BitsAndBytesConfig

# Set up 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    # Enable 4-bit model loading
    load_in_4bit=True,
    # Use double quantization to improve accuracy
    bnb_4bit_use_double_quant=True,
    # Apply NF4 (normal float 4-bit) quantization scheme
    bnb_4bit_quant_type="nf4",
    # Perform computations using bfloat16 precision
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the GPT-2 model using the specified quantization settings
model = AutoModelForCausalLM.from_pretrained("gpt2", quantization_config=quantization_config)

# Load a second GPT-2 model as a reference with the same quantization settings
model_ref = AutoModelForCausalLM.from_pretrained("gpt2", quantization_config=quantization_config)

# Initialize the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Use the end-of-sequence token as the padding token
tokenizer.pad_token = tokenizer.eos_token

# Pad sequences on the right to avoid FP16-related overflow issues
tokenizer.padding_side = "right"

# Turn off caching during the model's forward pass for training compatibility
model.config.use_cache = False


#4. Preprocess data set



The **"ultrafeedback\_binarized"** dataset on Hugging Face is a curated collection of **prompts paired with multiple model-generated responses**, where each response pair has been **binarized**—i.e., labeled to indicate which of the two responses is preferred. This makes it especially useful for training models with **Direct Preference Optimization (DPO)** or other **preference-based fine-tuning** methods.



In [ ]:
# Load the dataset from the specified location
ds = load_dataset("BarraHome/ultrafeedback_binarized")

This data set includes six splits.


In [ ]:
ds.keys()



Each record in the **"ultrafeedback\_binarized"** dataset contains multiple features, but the key ones to focus on are **"prompt"**, **"chosen"**, and **"rejected"**. For every prompt, the dataset provides two responses:

* **"chosen"**: the preferred response,
* **"rejected"**: the less preferred response.

This format is ideal for training models using **preference-based methods** like DPO, where learning is guided by comparing better and worse responses to the same prompt.


In [ ]:
ds["train_prefs"][0].keys()


You can examine a sample record from the dataset to see the key features: **"prompt"**, **"chosen"**, and **"rejected"** responses, along with several other metadata fields. These three core fields are essential for training models with preference-based techniques like DPO, as they clearly distinguish between a preferred and a less preferred response for each prompt.


In [ ]:
ds["train_prefs"][0]

Now, put the data set in the format that the DPO trainer accepts.

| Chosen | Rejected | Prompt |
| --- | --- | --- |
 | Developing a daily habit of drawing can be challenging <br>but with consistent practice, and a few tips. | One way to develop a habit of drawing daily is <br>to allocate a specific time interval for drawing. | How can I develop a habit of drawing daily?|


This code reduces the dataset size by selecting only the first 10% of each split to save computational resources. It defines and applies a function to clean the data by removing unused columns and extracting the actual text from nested response structures. Finally, it separates the processed data into training and evaluation sets for model training.


In [ ]:
# To conserve resources, reduce dataset size by keeping only the first 10% of each split
for key in ds:
    cnt = round(ds[key].__len__() * 0.10)
    ds[key] = ds[key].select(range(cnt))

# Define a function to clean and extract relevant fields from each data row
def process(row):
    # Remove unnecessary columns
    del row["prompt_id"]
    del row["messages"]
    del row["score_chosen"]
    del row["score_rejected"]
    # Extract the text content of the final message in chosen and rejected responses
    row["chosen"] = row["chosen"][-1]["content"]
    row["rejected"] = row["rejected"][-1]["content"]

    return row

# Apply the processing function to all splits using multiple CPU cores
ds = ds.map(
    process,
    num_proc=multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

# Assign the processed training and evaluation subsets
train_dataset = ds['train_prefs']
eval_dataset = ds['test_prefs']


Let's check the data record.


In [ ]:
train_dataset[0]

This code sets up a **LoRA (Low-Rank Adaptation)** configuration for **parameter-efficient fine-tuning (PEFT)** of a causal language model. It specifies which layers to adapt (`c_proj`, `c_attn`) and how, using a low-rank approximation with dropout and scaling. This approach enables efficient training by updating only a small subset of parameters rather than the full model.


In [ ]:
# PEFT (Parameter-Efficient Finetuning) configuration
peft_config = LoraConfig(
        # The rank of the low-rank adaptation weights
        r=4,
        # The target modules to apply the low-rank adaptation to
        target_modules=['c_proj','c_attn'],
        # The task type for the low-rank adaptation
        task_type="CAUSAL_LM",
        # The scaling factor for the low-rank adaptation weights
        lora_alpha=8,
        # The dropout probability for the low-rank adaptation weights
        lora_dropout=0.1,
        # The bias mode for the low-rank adaptation
        bias="none",
)

#5. DPO configuration

First, define trainThis code sets up the configuration for training a model using **Direct Preference Optimization (DPO)**. It defines training parameters such as batch size, learning rate, evaluation strategy, and the DPO-specific `beta` value, which controls the sharpness of preference learning. The `DPOConfig` is used later to guide the training process of a preference-based fine-tuning loop.


In [ ]:
# DPO configuration
from peft import get_peft_model
training_args = DPOConfig(
    # The beta parameter for the DPO loss function
    #beta is the temperature parameter for the DPO loss, typically something in the range of 0.1 to 0.5 .
    beta=0.1,
    # The output directory for the training
    output_dir="dpo",
    # The number of training epochs
    num_train_epochs=5,
    # The batch size per device during training
    per_device_train_batch_size=1,
    # The batch size per device during evaluation
    per_device_eval_batch_size=1,
    # Whether to remove unused columns from the dataset
    remove_unused_columns=False,
    # The number of steps between logging training progress
    logging_steps=10,
    # The number of gradient accumulation steps
    gradient_accumulation_steps=1,
    # The learning rate for the optimization
    learning_rate=1e-4,
    # The evaluation strategy (e.g., after each step or epoch)
    evaluation_strategy="epoch",
    # The number of warmup steps for the learning rate scheduler
    warmup_steps=2,
    # Whether to use 16-bit (float16) precision
    fp16=False,
    # The number of steps between saving checkpoints
    save_steps=500,
    # The maximum number of checkpoints to keep
    #save_total_limit=2,
    # The reporting backend to use (set to 'none' to disable, you can also report to wandb or tensorboard)
    report_to='none'
)

#6. DPO training

This code initializes a `DPOTrainer`, which is responsible for training the model using the **Direct Preference Optimization** method. It takes in the model, training configuration, tokenizer, datasets, and PEFT (LoRA-based) settings. Setting `tokenizer.pad_token = tokenizer.eos_token` ensures padding consistency, which is important for models like GPT-2 that do not have a default pad token.



In [ ]:
tokenizer.pad_token = tokenizer.eos_token

# Create a DPO trainer
# This trainer will handle the fine-tuning of the model using the DPO technique
trainer = DPOTrainer(
        # The model to be fine-tuned
        model=model,
        # The reference model (not used in this case because LoRA has been used)
        ref_model=None,
        # The DPO training configuration
        args=training_args,
        # The beta parameter for the DPO loss function

        # The training dataset
        train_dataset=train_dataset,
        # The evaluation dataset
        eval_dataset=eval_dataset,
        # The tokenizer for the model
        tokenizer=tokenizer,
        # The PEFT (Parallel Efficient Finetuning) configuration
        peft_config=peft_config,
        # The maximum prompt length
        #max_prompt_length=512,
        # The maximum sequence length
        max_length=512,
    )


Please note that when using LoRA for the base model, it's efficient to leave the model_ref param null, in which case the DPOTrainer will unload the adapter for reference inference.


Now, you're all set for training the model.


#7. Training model


In [ ]:
# Start the training process
trainer.train()

Let's retrieve and plot the training loss versus evaluation loss.


In [ ]:
# Retrieve log_history and save it to a dataframe
log = pd.DataFrame(trainer.state.log_history)
log_t = log[log['loss'].notna()]
log_e = log[log['eval_loss'].notna()]

# Plot train and evaluation losses
plt.plot(log_t["epoch"], log_t["loss"], label = "train_loss")
plt.plot(log_e["epoch"], log_e["eval_loss"], label = "eval_loss")
plt.legend()
plt.show()

In [ ]:
# Load the trained DPO model you just trained
dpo_model = AutoModelForCausalLM.from_pretrained('./dpo/checkpoint-250')


#8. Generation
This code sets a random seed for reproducibility and defines a text generation configuration using top-k sampling with low temperature for controlled output. A prompt is tokenized and passed to both the fine-tuned DPO model and the original GPT-2 model for response generation. The `generate()` method produces text based on the prompt and configuration. Finally, the generated outputs from both models are decoded and printed for comparison.


In [ ]:
# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

In [ ]:
# Set a seed for reproducibility
set_seed(42)


# Define the generation configuration for the DPO model
# This sets the parameters for text generation
generation_config = GenerationConfig(
        # Use sampling to generate diverse text
        do_sample=True,
        # Top-k sampling parameter
        top_k=1,
        # Temperature parameter to control the randomness of the generated text
        temperature=0.1,
        # Maximum number of new tokens to generate
        max_new_tokens=25,
        # Use the end-of-sequence token as the padding token
        pad_token_id=tokenizer.eos_token_id
    )

# Define the input prompt for text generation
PROMPT = "Is a higher octane gasoline better for your car?"
# Encode the prompt using the tokenizer
inputs = tokenizer(PROMPT, return_tensors='pt')

# Generate text using the DPO model
outputs = dpo_model.generate(**inputs, generation_config=generation_config)
# Decode the generated text and print it
print("DPO response:\t",tokenizer.decode(outputs[0], skip_special_tokens=True))

# Load the pre-trained GPT-2 model
gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2')
# Generate text using the GPT-2 model
outputs = gpt2_model.generate(**inputs, generation_config=generation_config)
# Decode the generated text and print it
print("\nGPT2 response:\t",tokenizer.decode(outputs[0], skip_special_tokens=True))

Althought the model is trained on a small data for 5 epochs only, it can be seen that the response generated by the DPO-tuned model is more concise and straightforward.
